## Weather stations to bronze

In [0]:
dbutils.widgets.text("file_name", "weather_stations.json")
FILE_NAME = dbutils.widgets.get("file_name")

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name
import json

date_utc = datetime.now(timezone.utc).strftime('%Y_%m_%d')

full_file_name = f"{FILE_NAME}_{date_utc}.json"

source_path = f"abfss://landing@stroutemindeuskadidev.dfs.core.windows.net/weather/stations/{full_file_name}"
delta_path = "abfss://bronze@stroutemindeuskadidev.dfs.core.windows.net/delta_tables/weather_stations/data"
deltaTable = "dbw_routemind_euskadi_dev.bronze.weather_stations"



df_weather_station_raw = spark.read \
    .option("multiline", "true") \
    .json(source_path)

In [0]:
df_weather_station_bronze = df_weather_station_raw \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file", input_file_name())

In [0]:

df_weather_station_bronze.write \
    .format("delta") \
    .option("path", delta_path) \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(deltaTable)


print(f"OVERWRITE completed on {deltaTable}. rows processed: {df_weather_station_bronze.count()}")


In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.air_quality_stations LIMIT 20

In [0]:
%sql
select * from dbw_routemind_euskadi_dev.bronze.weather_stations
LIMIT 20 